# Checked Merge Notebook — NileTTS Train QA Parts

Use this notebook after all 4 filtered train part files exist:

- `niletts_llm_qa_train_part_0.csv`
- `niletts_llm_qa_train_part_1.csv`
- `niletts_llm_qa_train_part_2.csv`
- `niletts_llm_qa_train_part_3.csv`

Important:

- Use the **filtered part files**, not the `checkpoint_...` files.
- The checkpoint files are useful only if you want to resume/re-filter a part.
- This notebook merges, validates, normalizes labels, removes duplicates, and saves the final train QA CSV.

### ======
### Final training-data preparation

This notebook reads the four filtered training QA parts directly.
It does not depend on the output of the earlier merge notebook.

Final output:
`1_train_data/niletts_llm_qa_train.csv`
### ======

In [ ]:
%pip install "numpy==1.26.4" "pandas==2.2.2"

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
EXPECTED_TRAIN_SOURCE_ROWS = 7713


## 1. Load the four filtered part files

In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "2_QA_GENERATION").is_dir():
        QA_DIR = candidate / "2_QA_GENERATION"
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )
# =====


In [ ]:
# =====
# part_files = [
#     "niletts_llm_qa_train_part_0.csv",
#     "niletts_llm_qa_train_part_1.csv",
#     "niletts_llm_qa_train_part_2.csv",
#     "niletts_llm_qa_train_part_3.csv",
# ]

part_files = [
    QA_DIR / "1_train_data" / "filtered"
    / f"niletts_llm_qa_train_part_{part_id}.csv"
    for part_id in range(4)
]
# =====

missing = [f for f in part_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

dfs = []
for f in part_files:
    df = pd.read_csv(f)
    print(f, df.shape)
    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

print("Merged raw shape:", merged_df.shape)
display(merged_df.head())


## 2. Validate required columns

In [ ]:
required_cols = [
    "id",
    "split",
    "part_id",
    "metadata_row_index",
    "qa_train_source_index",
    "audio_file",
    "speaker_name",
    "domain",
    "context_ref",
    "question",
    "answer",
    "answer_start_ref",
    "valid_answer_span_ref",
    "question_type",
    "difficulty",
]

missing_cols = [c for c in required_cols if c not in merged_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in merged data: {missing_cols}")

print("All required columns exist.")


## 3. Clean text, re-check answer spans, normalize labels

In [ ]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


def normalize_question_type(x):
    x = normalize_spaces(x).lower()

    allowed = {
        "main_idea",
        "detail_extraction",
        "intent",
        "domain_specific",
        "number_extraction",
    }

    if x in allowed:
        return x

    # Fix local LLM outputs like:
    # "main_idea/detail_extraction" or
    # "main_idea/detail_extraction/intent/domain_specific/number_extraction"
    parts = [p.strip() for p in x.split("/") if p.strip()]

    for preferred in [
        "domain_specific",
        "detail_extraction",
        "intent",
        "main_idea",
        "number_extraction",
    ]:
        if preferred in parts:
            return preferred

    if "price" in x or "number" in x:
        return "number_extraction"

    return "other"


def normalize_difficulty(x):
    x = normalize_spaces(x).lower()

    if x in {"easy", "medium"}:
        return x

    # The generation prompt asked for easy/medium.
    # If local Qwen returned hard, map it to medium.
    if x == "hard":
        return "medium"

    return "medium"


clean_df = merged_df.copy()

for col in ["question", "answer", "context_ref", "audio_file", "speaker_name", "domain"]:
    clean_df[col] = clean_df[col].astype(str).map(normalize_spaces)

clean_df["answer_start_ref"] = clean_df.apply(
    lambda r: find_answer_start_soft(r["context_ref"], r["answer"]),
    axis=1
)

clean_df["valid_answer_span_ref"] = clean_df["answer_start_ref"] != -1

clean_df["question_type_raw"] = clean_df["question_type"].astype(str)
clean_df["difficulty_raw"] = clean_df["difficulty"].astype(str)

clean_df["question_type"] = clean_df["question_type"].map(normalize_question_type)
clean_df["difficulty"] = clean_df["difficulty"].map(normalize_difficulty)

before = len(clean_df)

clean_df = clean_df[clean_df["question"].str.len() > 5]
clean_df = clean_df[clean_df["answer"].str.len() > 0]
clean_df = clean_df[clean_df["valid_answer_span_ref"] == True]

clean_df = clean_df.drop_duplicates(
    subset=["audio_file", "question", "answer"]
).reset_index(drop=True)

print("Rows before cleaning:", before)
print("Rows after cleaning:", len(clean_df))
print("Removed rows:", before - len(clean_df))


## 4. Coverage checks

In [ ]:
covered_source_rows = clean_df["qa_train_source_index"].nunique()
missing_count = EXPECTED_TRAIN_SOURCE_ROWS - covered_source_rows

covered_indices = set(clean_df["qa_train_source_index"].astype(int).tolist())
missing_indices = sorted(list(set(range(EXPECTED_TRAIN_SOURCE_ROWS)) - covered_indices))

coverage_report = pd.DataFrame({
    "metric": [
        "expected_train_source_rows",
        "covered_train_source_rows",
        "missing_train_source_rows",
        "total_qa_pairs_after_cleaning",
        "average_qa_pairs_per_covered_transcript",
    ],
    "value": [
        EXPECTED_TRAIN_SOURCE_ROWS,
        covered_source_rows,
        missing_count,
        len(clean_df),
        len(clean_df) / max(1, covered_source_rows),
    ],
})

display(coverage_report)

print("First 30 missing source indices:")
print(missing_indices[:30])


## 5. Distribution checks

In [ ]:
print("Rows per part:")
display(clean_df["part_id"].value_counts().sort_index())

print("QA pairs per domain:")
display(clean_df["domain"].value_counts())

print("QA pairs per question type:")
display(clean_df["question_type"].value_counts(dropna=False))

print("QA pairs per difficulty:")
display(clean_df["difficulty"].value_counts(dropna=False))

sample_cols = [
    "qa_train_source_index",
    "part_id",
    "audio_file",
    "domain",
    "question_type",
    "difficulty",
    "question",
    "answer",
    "context_ref",
]

display(clean_df[sample_cols].sample(
    min(10, len(clean_df)),
    random_state=SEED
))


## 6. Shuffle and save final train QA file

In [ ]:
final_df = clean_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# =====
# final_output = "niletts_llm_qa_train.csv"
final_output = QA_DIR / "1_train_data" / "niletts_llm_qa_train.csv"
# =====

final_df.to_csv(
    final_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", final_output)
print("Final shape:", final_df.shape)

display(final_df.head())


## 7. Save missing source indices

If you want to generate additional training coverage later, use this file to identify missing transcript rows.


In [ ]:
missing_df = pd.DataFrame({"qa_train_source_index": missing_indices})

# =====
# missing_output = "missing_qa_train_source_indices_checked.csv"
missing_output = (
    QA_DIR / "1_train_data" / "missing"
    / "missing_qa_train_source_indices_checked.csv"
)
# =====

missing_df.to_csv(
    missing_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", missing_output)
display(missing_df.head())
